# Profiling

Build a measurement workflow that moves from symptoms to evidence about bottlenecks.

## Objectives

Define a reproducible workload, collect appropriate evidence, and avoid perturbing or over-interpreting measurements.

## Background

Profilers add overhead and expose different layers of a system; useful conclusions require a stable baseline and a focused question.

## Prediction

The first workload will contain three explicit phases:

1. transform a sequence of integers using repeated arithmetic and bitwise operations;
2. build a histogram from the transformed values;
3. calculate a weighted checksum.

All three phases traverse the same number of elements, but the transformation performs several rounds of arithmetic per element. It should therefore account for the largest fraction of unprofiled runtime.

The histogram and checksum phases should remain visible but secondary. Their relative cost cannot be inferred from operation counts alone because Python integer arithmetic, list access, loop mechanics, and memory allocation all contribute to elapsed time.

Before collecting a profile, the workload must have:

- deterministic input and output;
- a bounded runtime;
- fixed CPU affinity;
- warm-up runs;
- repeated baseline measurements.

A deterministic function-level profiler such as `cProfile` should later attribute most cumulative time to the transformation phase. However, the profiled runtime should be longer than the unprofiled baseline because profiling records Python call events. The profile can therefore identify where observed runtime is attributed, but its timing should not be treated as an unperturbed measurement of production performance.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [ ]:
import os
from contextlib import contextmanager
from pprint import pprint

import pandas as pd

from common.benchmark import benchmark_callable


PROFILE_CPU = 15

print(f"Available CPUs: {sorted(os.sched_getaffinity(0))}")
print(f"Selected profiling CPU: {PROFILE_CPU}")

if PROFILE_CPU not in os.sched_getaffinity(0):
    raise RuntimeError(f"CPU {PROFILE_CPU} is not available to the current process")


@contextmanager
def pinned_to_cpu(cpu_id: int):
    original_affinity = os.sched_getaffinity(0)

    try:
        os.sched_setaffinity(0, {cpu_id})
        yield
    finally:
        os.sched_setaffinity(0, original_affinity)


with pinned_to_cpu(PROFILE_CPU):
    print(f"Temporary affinity: {sorted(os.sched_getaffinity(0))}")

Available CPUs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Selected profiling CPU: 15
Temporary affinity: [15]


### Controlled profiling workload

The workload below is intentionally divided into named Python functions. This will later allow a function-level profiler to attribute time to meaningful phases rather than to one monolithic function.

The input is constructed once and reused. Each measured invocation:

1. allocates and fills a transformed list;
2. builds a 256-bin histogram;
3. calculates a weighted checksum.

The returned values prevent the final results from being semantically unused and provide simple correctness checks.

This is not intended to model a useful application. It is a controlled workload for learning how baseline timing and profile evidence relate.

In [ ]:
ELEMENT_COUNT = 1_000_000
TRANSFORM_ROUNDS = 3

UINT32_MASK = (1 << 32) - 1
UINT64_MASK = (1 << 64) - 1

input_values = [
    ((index * 2_654_435_761) ^ (index >> 3)) & UINT32_MASK
    for index in range(ELEMENT_COUNT)
]

workload_configuration = {
    "element_count": ELEMENT_COUNT,
    "transform_rounds": TRANSFORM_ROUNDS,
    "histogram_bins": 256,
    "input_size_mib_estimate": (input_values.__sizeof__() / 1024**2),
}

pprint(workload_configuration)

{'element_count': 1000000,
 'histogram_bins': 256,
 'input_size_mib_estimate': 8.057319641113281,
 'transform_rounds': 3}


In [ ]:
def transform_values(
    values: list[int],
    rounds: int,
) -> list[int]:
    transformed = [0] * len(values)

    for index, value in enumerate(values):
        current = value

        for _ in range(rounds):
            current ^= current >> 16
            current = (current * 0x45D9F3B) & UINT32_MASK
            current ^= current >> 16

        transformed[index] = current

    return transformed


def build_low_byte_histogram(values: list[int]) -> list[int]:
    histogram = [0] * 256

    for value in values:
        histogram[value & 0xFF] += 1

    return histogram


def calculate_weighted_checksum(values: list[int]) -> int:
    checksum = 0

    for index, value in enumerate(values, start=1):
        checksum = (checksum + index * (value & 0xFFFF)) & UINT64_MASK

    return checksum


def profiling_workload() -> tuple[int, int, int]:
    transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    histogram = build_low_byte_histogram(transformed)
    checksum = calculate_weighted_checksum(transformed)

    return (
        checksum,
        sum(histogram),
        max(histogram),
    )

### Correctness and determinism check

Before timing the workload, run it twice and verify that:

- every transformed element is represented in the histogram;
- repeated executions produce the same result.

These executions also provide initial interpreter and memory-allocation warm-up, but they are not part of the measured baseline.

In [5]:
with pinned_to_cpu(PROFILE_CPU):
    first_result = profiling_workload()
    second_result = profiling_workload()

assert first_result == second_result
assert first_result[1] == ELEMENT_COUNT
assert 0 < first_result[2] <= ELEMENT_COUNT

correctness_result = {
    "checksum": first_result[0],
    "histogram_total": first_result[1],
    "largest_histogram_bin": first_result[2],
    "deterministic": first_result == second_result,
}

pprint(correctness_result)

{'checksum': 16378977129066911,
 'deterministic': True,
 'histogram_total': 1000000,
 'largest_histogram_bin': 4112}


### Unprofiled baseline

The baseline uses wall-clock time without an active profiler. All warm-up and measured iterations run while the notebook process is pinned to CPU 15.

Individual trials are retained because the distribution matters. A single minimum or mean cannot show scheduling noise, thermal effects, frequency changes, or occasional operating-system interference.

This baseline will later be compared with the runtime of the same workload under profiling.

In [ ]:
BASELINE_WARMUP_ITERATIONS = 1
BASELINE_ITERATIONS = 7

with pinned_to_cpu(PROFILE_CPU):
    baseline_result = benchmark_callable(
        profiling_workload,
        warmup_iterations=BASELINE_WARMUP_ITERATIONS,
        iterations=BASELINE_ITERATIONS,
    )

baseline_trials = pd.DataFrame(
    {
        "trial": range(1, baseline_result.iterations + 1),
        "wall_ms": [
            duration_ns / 1_000_000 for duration_ns in baseline_result.durations_ns
        ],
    }
)

baseline_trials.round(3)

,trial,wall_ms
0,1,470.828
1,2,470.727
2,3,470.911
3,4,469.696
4,5,470.071
5,6,468.926
6,7,469.325


In [ ]:
baseline_summary = pd.DataFrame(
    [
        {
            "iterations": baseline_result.iterations,
            "minimum_ms": baseline_result.minimum_ns / 1_000_000,
            "median_ms": baseline_result.median_ns / 1_000_000,
            "mean_ms": baseline_result.mean_ns / 1_000_000,
            "maximum_ms": baseline_result.maximum_ns / 1_000_000,
            "standard_deviation_ms": (
                baseline_result.standard_deviation_ns / 1_000_000
            ),
            "relative_standard_deviation_percent": (
                100 * baseline_result.standard_deviation_ns / baseline_result.mean_ns
            ),
        }
    ]
)

baseline_summary.round(3)

,iterations,minimum_ms,median_ms,mean_ms,maximum_ms,standard_deviation_ms,relative_standard_deviation_percent
0,7,468.926,470.071,470.069,470.911,0.729,0.155


### Deterministic function-level profile

`cProfile` is a deterministic profiler: it observes Python call and return events and accumulates timing statistics for functions encountered during execution.

We will collect several independent profiles while retaining the wall-clock duration of each profiled run. This permits two separate comparisons:

1. **profile attribution**: which functions account for the observed profiled runtime;
2. **profiler perturbation**: how much slower the workload becomes while profiling is active.

The profiler is enabled immediately before `profiling_workload()` and disabled immediately afterward. CPU affinity remains fixed at CPU 15.

The profiler's function times are measurements from the instrumented execution. They should not be substituted for the unprofiled baseline.

In [8]:
import cProfile
import pstats
from time import perf_counter_ns


PROFILE_ITERATIONS = 5


def collect_cprofile_trial() -> tuple[
    tuple[int, int, int],
    cProfile.Profile,
    int,
]:
    profiler = cProfile.Profile()

    start_ns = perf_counter_ns()
    profiler.enable()

    try:
        result = profiling_workload()
    finally:
        profiler.disable()
        wall_ns = perf_counter_ns() - start_ns

    return result, profiler, wall_ns

In [9]:
profiled_runs = []

with pinned_to_cpu(PROFILE_CPU):
    for trial in range(1, PROFILE_ITERATIONS + 1):
        result, profiler, wall_ns = collect_cprofile_trial()

        assert result == first_result

        profiled_runs.append(
            {
                "trial": trial,
                "result": result,
                "profiler": profiler,
                "wall_ns": wall_ns,
            }
        )

profiled_trials = pd.DataFrame(
    [
        {
            "trial": run["trial"],
            "wall_ms": run["wall_ns"] / 1_000_000,
        }
        for run in profiled_runs
    ]
)

profiled_trials.round(3)

,trial,wall_ms
0,1,483.402
1,2,490.169
2,3,492.176
3,4,480.899
4,5,492.146


In [ ]:
profiled_median_ns = float(profiled_trials["wall_ms"].median()) * 1_000_000
baseline_median_ns = baseline_result.median_ns

profiling_overhead_summary = pd.DataFrame(
    [
        {
            "baseline_median_ms": baseline_median_ns / 1_000_000,
            "profiled_median_ms": profiled_median_ns / 1_000_000,
            "added_median_ms": (profiled_median_ns - baseline_median_ns) / 1_000_000,
            "slowdown_factor": (profiled_median_ns / baseline_median_ns),
            "overhead_percent": (
                100 * (profiled_median_ns - baseline_median_ns) / baseline_median_ns
            ),
        }
    ]
)

profiling_overhead_summary.round(3)

,baseline_median_ms,profiled_median_ms,added_median_ms,slowdown_factor,overhead_percent
0,470.071,490.169,20.097,1.043,4.275


### Selecting a representative profile

The profile whose wall time is closest to the median profiled wall time is used for function-level inspection.

Selecting the median run avoids presenting an unusually fast or slow trial as representative. It does not combine function statistics across runs: the displayed function times all come from one internally consistent execution.

In [ ]:
profiled_trials["distance_from_median_ms"] = (
    profiled_trials["wall_ms"] - profiled_trials["wall_ms"].median()
).abs()

representative_trial_index = int(profiled_trials["distance_from_median_ms"].idxmin())
representative_run = profiled_runs[representative_trial_index]

representative_selection = {
    "trial": representative_run["trial"],
    "wall_ms": representative_run["wall_ns"] / 1_000_000,
    "median_profiled_wall_ms": profiled_trials["wall_ms"].median(),
}

pprint(representative_selection)

{'median_profiled_wall_ms': np.float64(490.16875),
 'trial': 2,
 'wall_ms': 490.16875}


### Function-level statistics

For each function, `cProfile` records:

- **primitive calls**: calls not induced by recursion;
- **total calls**: all calls, including recursive calls;
- **internal time** (`tottime`): time spent in the function body, excluding callees;
- **cumulative time** (`cumtime`): time spent in the function and functions it called.

The workload phases do not call one another, so their internal and cumulative times should be similar. `profiling_workload`, however, calls all three phases; its cumulative time should cover nearly the entire instrumented workload while its internal time should remain small.

In [ ]:
representative_stats = pstats.Stats(representative_run["profiler"])

profile_rows = []

for function_key, statistics in representative_stats.stats.items():
    filename, line_number, function_name = function_key
    primitive_calls, total_calls, internal_s, cumulative_s, _ = statistics

    profile_rows.append(
        {
            "function": function_name,
            "filename": Path(filename).name,
            "line": line_number,
            "primitive_calls": primitive_calls,
            "total_calls": total_calls,
            "internal_ms": internal_s * 1_000,
            "cumulative_ms": cumulative_s * 1_000,
        }
    )

function_profile = (
    pd.DataFrame(profile_rows)
    .sort_values(
        ["cumulative_ms", "internal_ms"],
        ascending=False,
    )
    .reset_index(drop=True)
)

function_profile.head(15).round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms
0,profiling_workload,1169694572.py,40,1,1,0.013,480.672
1,transform_values,1169694572.py,1,1,1,388.478,388.478
2,calculate_weighted_checksum,1169694572.py,29,1,1,63.785,63.785
3,build_low_byte_histogram,1169694572.py,20,1,1,28.392,28.392
4,<method 'disable' of '_lsprof.Profiler' objects>,~,0,1,1,0.057,0.057
5,<built-in method builtins.sum>,~,0,1,1,0.002,0.002
6,<built-in method builtins.max>,~,0,1,1,0.002,0.002
7,<built-in method builtins.len>,~,0,1,1,0.000,0.000


In [ ]:
WORKLOAD_FUNCTIONS = {
    "profiling_workload",
    "transform_values",
    "build_low_byte_histogram",
    "calculate_weighted_checksum",
}

workload_function_profile = (
    function_profile[function_profile["function"].isin(WORKLOAD_FUNCTIONS)]
    .copy()
    .sort_values("cumulative_ms", ascending=False)
    .reset_index(drop=True)
)

representative_wall_ms = representative_run["wall_ns"] / 1_000_000

workload_function_profile["internal_fraction_percent"] = (
    100 * workload_function_profile["internal_ms"] / representative_wall_ms
)

workload_function_profile.round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms,internal_fraction_percent
0,profiling_workload,1169694572.py,40,1,1,0.013,480.672,0.003
1,transform_values,1169694572.py,1,1,1,388.478,388.478,79.254
2,calculate_weighted_checksum,1169694572.py,29,1,1,63.785,63.785,13.013
3,build_low_byte_histogram,1169694572.py,20,1,1,28.392,28.392,5.792


### Minimally instrumented phase timing

The deterministic profile attributed most runtime to `transform_values`, followed by the checksum and histogram phases. Those measurements were collected while `cProfile` was active.

To test whether profiling materially changes the relative phase distribution, the workload is repeated with explicit wall-clock timestamps around the three phase boundaries.

This instrumentation adds only a small fixed number of timer calls per workload invocation. It still perturbs execution and does not expose activity inside each phase, but it provides an independent timing method with substantially less instrumentation than deterministic call profiling.

In [15]:
PHASE_TIMING_ITERATIONS = 7


def timed_profiling_workload() -> dict[str, int]:
    workload_start_ns = perf_counter_ns()

    transform_start_ns = workload_start_ns
    transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    transform_end_ns = perf_counter_ns()

    histogram = build_low_byte_histogram(transformed)
    histogram_end_ns = perf_counter_ns()

    checksum = calculate_weighted_checksum(transformed)
    checksum_end_ns = perf_counter_ns()

    result = (
        checksum,
        sum(histogram),
        max(histogram),
    )

    assert result == first_result

    return {
        "transform_ns": transform_end_ns - transform_start_ns,
        "histogram_ns": histogram_end_ns - transform_end_ns,
        "checksum_ns": checksum_end_ns - histogram_end_ns,
        "workload_ns": checksum_end_ns - workload_start_ns,
    }

In [ ]:
phase_measurements = []

with pinned_to_cpu(PROFILE_CPU):
    timed_profiling_workload()

    for trial in range(1, PHASE_TIMING_ITERATIONS + 1):
        measurement = timed_profiling_workload()
        measurement["trial"] = trial
        phase_measurements.append(measurement)

phase_trials = pd.DataFrame(phase_measurements)

for column in (
    "transform_ns",
    "histogram_ns",
    "checksum_ns",
    "workload_ns",
):
    phase_trials[column.removesuffix("_ns") + "_ms"] = phase_trials[column] / 1_000_000

phase_trials[
    [
        "trial",
        "transform_ms",
        "histogram_ms",
        "checksum_ms",
        "workload_ms",
    ]
].round(3)

,trial,transform_ms,histogram_ms,checksum_ms,workload_ms
0,1,359.349,30.289,62.012,451.650
1,2,359.822,30.957,62.815,453.594
2,3,360.001,30.896,61.902,452.799
3,4,359.862,30.426,61.845,452.133
4,5,359.388,28.465,62.471,450.323
5,6,359.709,30.440,62.109,452.258
6,7,359.288,28.429,61.649,449.366


In [17]:
phase_summary = pd.DataFrame(
    [
        {
            "phase": "transform_values",
            "median_ms": phase_trials["transform_ms"].median(),
        },
        {
            "phase": "build_low_byte_histogram",
            "median_ms": phase_trials["histogram_ms"].median(),
        },
        {
            "phase": "calculate_weighted_checksum",
            "median_ms": phase_trials["checksum_ms"].median(),
        },
    ]
)

median_phase_total_ms = phase_summary["median_ms"].sum()

phase_summary["fraction_percent"] = (
    100 * phase_summary["median_ms"] / median_phase_total_ms
)

phase_summary.round(3)

,phase,median_ms,fraction_percent
0,transform_values,359.709,79.556
1,build_low_byte_histogram,30.426,6.729
2,calculate_weighted_checksum,62.012,13.715


In [18]:
cprofile_phase_comparison = (
    workload_function_profile[
        workload_function_profile["function"] != "profiling_workload"
    ][
        [
            "function",
            "internal_ms",
            "internal_fraction_percent",
        ]
    ]
    .rename(
        columns={
            "internal_ms": "cprofile_internal_ms",
            "internal_fraction_percent": "cprofile_fraction_percent",
        }
    )
    .merge(
        phase_summary.rename(columns={"phase": "function"}),
        on="function",
        how="inner",
    )
)

cprofile_phase_comparison["cprofile_to_manual_time_ratio"] = (
    cprofile_phase_comparison["cprofile_internal_ms"]
    / cprofile_phase_comparison["median_ms"]
)

cprofile_phase_comparison["fraction_difference_percentage_points"] = (
    cprofile_phase_comparison["cprofile_fraction_percent"]
    - cprofile_phase_comparison["fraction_percent"]
)

cprofile_phase_comparison.round(3)

,function,cprofile_internal_ms,cprofile_fraction_percent,median_ms,fraction_percent,cprofile_to_manual_time_ratio,fraction_difference_percentage_points
0,transform_values,388.478,79.254,359.709,79.556,1.080,-0.302
1,calculate_weighted_checksum,63.785,13.013,62.012,13.715,1.029,-0.702
2,build_low_byte_histogram,28.392,5.792,30.426,6.729,0.933,-0.937


In [19]:
phase_timing_overhead = pd.DataFrame(
    [
        {
            "baseline_median_ms": baseline_result.median_ns / 1_000_000,
            "phase_timed_median_ms": phase_trials["workload_ms"].median(),
            "slowdown_factor": (
                phase_trials["workload_ms"].median()
                / (baseline_result.median_ns / 1_000_000)
            ),
            "overhead_percent": (
                100
                * (
                    phase_trials["workload_ms"].median()
                    - baseline_result.median_ns / 1_000_000
                )
                / (baseline_result.median_ns / 1_000_000)
            ),
        }
    ]
)

phase_timing_overhead.round(3)

,baseline_median_ms,phase_timed_median_ms,slowdown_factor,overhead_percent
0,470.071,452.133,0.962,-3.816


## Observations

TODO: Record baseline timing, profiler overhead, and evidence from actual runs.

## Explanation

TODO: Explain the bottleneck using profile evidence and state the tool's limitations.

## Connection to LLMs

Profiling separates compute, memory, synchronization, input, and communication bottlenecks in LLM systems.

## Further Exploration

TODO: Form a single optimization hypothesis and define a before/after measurement.